# اليوم الأول — معالجة النصوص والترميز
## Day 1 — Text Processing & Tokenisation

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار / Track:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/01_text_processing_tokenization.ipynb)

> السؤال المحوري: كيف يتحول نص عربي أو إنجليزي إلى تمثيل عددي صالح للنموذج دون إفساد معناه؟
>
> Driving question: How does Arabic or English text become a model-ready numeric representation without damaging its meaning?

**بنهاية المختبر ستتمكن من / By the end, you can:**

1. فحص Unicode بدل التخمين — inspect Unicode rather than guess.
2. بناء نسختين: نص خام محفوظ ونص مهيأ للنموذج — keep raw and model-ready copies.
3. قياس **خصوبة الترميز Token Fertility** ومعدل القطع — measure fertility and truncation.
4. تحويل Token IDs إلى Embeddings بسيطة — map token IDs to simple embeddings.
5. حفظ دليل إنجاز قابل للتسليم عبر GitHub — save GitHub-ready evidence.

## 0) طريقة العمل / How to work

- نفّذ الخلايا بالترتيب: **Runtime → Core → Checkpoint**.
- نتيجة كل خلية موضحة بجملة تبدأ بـ **المتوقع / Expected**.
- إذا تعذر تنزيل نموذج Hugging Face، أكمل بالمرمّز المحلي؛ فهو جزء أساسي يعمل دون شبكة بعد تثبيت الحزم.
- لا تستخدم بيانات حقيقية أو أسماء أو أرقام أشخاص. البيانات أدناه اصطناعية.

**المتوقع / Expected:** في النهاية تظهر العبارة `DAY1_NOTEBOOK1_CORE=PASS`.

In [ ]:
# إعداد بيئة قابلة لإعادة التشغيل / Reproducible setup
import importlib.util
import subprocess
import sys

PINNED = {"transformers": "5.15.1", "tokenizers": "0.22.2", "spacy": "3.8.7"}
missing = [name for name in PINNED if importlib.util.find_spec(name) is None]
if missing:
    packages = [f"{name}=={PINNED[name]}" for name in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

Python: 3.13.15
Environment ready / البيئة جاهزة


### لماذا نثبت نسخًا محددة؟ / Why pin versions?

لتقليل اختلاف النتائج بين أجهزة المتدربين. التثبيت مجاني ولا يحتاج اشتراكًا؛ يحتاج اتصالًا بالإنترنت أول مرة فقط. Google Colab المجاني يكفي لهذا اليوم، ولا يضمن GPU أو مدة جلسة ثابتة.

**المتوقع / Expected:** إصدار Python ثم رسالة نجاح البيئة.

In [ ]:
import html
import re
import unicodedata
from dataclasses import dataclass
from typing import Iterable

import numpy as np
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing

SEED = 42
rng = np.random.default_rng(SEED)
samples = [
    "أهلاً وسهلاً بكم في برنامج بيان!",
    "مرحبــاً\u00a0بكم",
    "Contact us at learner@example.org",
    "للتجربة فقط: 0551234567",
    "Natural language processing connects text and models.",
]
print("Synthetic samples:", len(samples))

Synthetic samples: 5


## 1) Unicode قبل التنظيف / Unicode before cleaning

الحرف الظاهر قد يتكون من نقطة ترميز واحدة أو أكثر. لا تخلط بين:

- **Unicode code point:** رقم معياري للحرف.
- **UTF-8:** تحويل نقاط الترميز إلى bytes للتخزين والنقل.
- **Token ID:** رقم داخل قاموس مرمّز محدد، وليس رقم Unicode.

نستخدم **NFC** لتوحيد التمثيلات المتكافئة غالبًا، لكن لا نحول العربية بقوة من دون سبب متعلق بالمهمة.

In [ ]:
def inspect_unicode(text: str) -> list[dict[str, str]]:
    return [
        {"char": ch, "code_point": f"U+{ord(ch):04X}", "name": unicodedata.name(ch, "UNKNOWN")}
        for ch in text
    ]

unicode_rows = inspect_unicode("أé")
for row in unicode_rows:
    print(row)
assert unicode_rows[0]["code_point"] == "U+0623"
print("Unicode inspection=PASS")

{'char': 'أ', 'code_point': 'U+0623', 'name': 'ARABIC LETTER ALEF WITH HAMZA ABOVE'}
{'char': 'é', 'code_point': 'U+00E9', 'name': 'LATIN SMALL LETTER E WITH ACUTE'}
Unicode inspection=PASS


**المتوقع / Expected:** صف لكل حرف يحوي الشكل و`U+....` والاسم، ثم رسالة نجاح.

### قرار عربي محافظ / Conservative Arabic decision

ابدأ بأقل تعديل ممكن. إزالة التشكيل أو توحيد الألف قد يفيد البحث أو التصنيف، لكنه قد يضر مهامًا تعتمد الشكل الأصلي. لذلك نسجل الإعدادات ولا نطبقها بصمت.

In [ ]:
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"
WHITESPACE = re.compile(r"\s+")
HTML_TAG = re.compile(r"<[^>]+>")
EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
SAUDI_MOBILE = re.compile(r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)")

@dataclass(frozen=True)
class TextRecord:
    raw_text: str
    model_text: str


def mask_pii(text: str) -> str:
    return SAUDI_MOBILE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))


def prepare_text(text: str, *, remove_diacritics: bool = False, normalize_alef: bool = False) -> TextRecord:
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    raw = text
    model = unicodedata.normalize("NFC", html.unescape(text))
    model = HTML_TAG.sub(" ", model).replace(TATWEEL, "")
    if remove_diacritics:
        model = ARABIC_DIACRITICS.sub("", model)
    if normalize_alef:
        model = re.sub(r"[إأآٱ]", "ا", model)
    model = WHITESPACE.sub(" ", mask_pii(model)).strip()
    return TextRecord(raw_text=raw, model_text=model)

records = [prepare_text(text) for text in samples]
for record in records:
    print({"raw": record.raw_text, "model": record.model_text})
assert records[1].raw_text != records[1].model_text
assert "learner@example.org" not in records[2].model_text
assert "0551234567" not in records[3].model_text
assert records[2].raw_text == samples[2]
print("Two-copy preprocessing contract=PASS")

{'raw': 'أهلاً وسهلاً بكم في برنامج بيان!', 'model': 'أهلاً وسهلاً بكم في برنامج بيان!'}
{'raw': 'مرحبــاً\xa0بكم', 'model': 'مرحباً بكم'}
{'raw': 'Contact us at learner@example.org', 'model': 'Contact us at <EMAIL>'}
{'raw': 'للتجربة فقط: 0551234567', 'model': 'للتجربة فقط: <PHONE>'}
{'raw': 'Natural language processing connects text and models.', 'model': 'Natural language processing connects text and models.'}
Two-copy preprocessing contract=PASS


**المتوقع / Expected:** تبقى `raw` كما دخلت، بينما تحذف الكشيدة وتوحد المسافات وتستبدل البريد والهاتف بعلامات عامة.

> **قاعدة البيانات:** النسخة الخام للتدقيق وإعادة المعالجة، والنسخة المهيأة لدخول النموذج. لا تُنشر النسخة الخام إن احتوت بيانات شخصية.

### جرّب / Try

غيّر `remove_diacritics` و`normalize_alef` على مثال اصطناعي، ثم اكتب هل التغيير يخدم مهمتك أم يخفي فرقًا مهمًا.

## 2) spaCy كمرحلة قابلة للاختبار / A testable spaCy stage

التقسيم إلى جمل لا يُنفذ بـ`split('.')`. نمرر **نسخة النموذج المحمية** إلى spaCy، ونوثق قيدًا معروفًا: الاختصارات مثل `د.` تحتاج استثناءً أو segmenter مدربًا قبل الإنتاج. هذا المثال يستخدم `sentencizer` القاعدي بلا تنزيل نموذج لغوي.

In [ ]:
import spacy

sentence_nlp = spacy.blank("xx")
sentence_nlp.add_pipe("sentencizer")

def split_model_sentences(text: str) -> list[str]:
    protected = prepare_text(text).model_text
    return [sentence.text.strip() for sentence in sentence_nlp(protected).sents if sentence.text.strip()]

sentence_examples = {
    "ar": "الخدمة جيدة. لم يصل الرمز!",
    "en": "The portal stopped. Please retry.",
}
sentence_splits = {language: split_model_sentences(text) for language, text in sentence_examples.items()}
for language, sentences in sentence_splits.items():
    print(language, sentences)

abbreviation_probe = split_model_sentences("راجع د. أحمد. ثم أعد المحاولة.")
print("Known abbreviation probe / حالة تحتاج قاعدة إضافية:", abbreviation_probe)
assert sentence_splits["ar"] == ["الخدمة جيدة.", "لم يصل الرمز!"]
assert sentence_splits["en"] == ["The portal stopped.", "Please retry."]
print("SPACY_SENTENCE_PIPELINE=PASS")

ar ['الخدمة جيدة.', 'لم يصل الرمز!']
en ['The portal stopped.', 'Please retry.']
Known abbreviation probe / حالة تحتاج قاعدة إضافية: ['راجع د.', 'أحمد.', 'ثم أعد المحاولة.']
SPACY_SENTENCE_PIPELINE=PASS


In [ ]:
# Distinction: Saudi dialect slice without changing the original text

dialect_text = "الخدمة مره ممتازة وما تأخرت"

dialect_prepared = prepare_text(dialect_text)
dialect_sentences = split_model_sentences(dialect_text)

print("Original text:", dialect_text)
print("Model text:", dialect_prepared.model_text)
print("Sentences:", dialect_sentences)

assert dialect_text == "الخدمة مره ممتازة وما تأخرت"
print("Saudi dialect slice=PASS")

Original text: الخدمة مره ممتازة وما تأخرت
Model text: الخدمة مره ممتازة وما تأخرت
Sentences: ['الخدمة مره ممتازة وما تأخرت']
Saudi dialect slice=PASS


## 2) الترميز إلى كلمات فرعية / Subword tokenisation

المرمّز المحلي التالي **تعليمي ومحدود**: يوضح الرموز الخاصة و`[UNK]` وToken IDs دون تنزيل نموذج. ليس بديلًا عن مرمّز النموذج النهائي. في Explore سنقارن اختياريًا مع mBERT.

In [ ]:
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
known_tokens = [
    "أهلاً", "وسهلاً", "بكم", "في", "برنامج", "بيان", "!", "مرحبا",
    "Contact", "us", "at", "<", "EMAIL", ">", "Natural", "language",
    "processing", "connects", "text", "and", "models", ".",
]
vocab_tokens = special_tokens + known_tokens
vocab = {token: idx for idx, token in enumerate(vocab_tokens)}
local_tokenizer = Tokenizer(WordPiece(vocab=vocab, unk_token="[UNK]"))
local_tokenizer.pre_tokenizer = BertPreTokenizer()
local_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[("[CLS]", vocab["[CLS]"]), ("[SEP]", vocab["[SEP]"])],
)
for record in records:
    encoded = local_tokenizer.encode(record.model_text)
    print(record.model_text)
    print("tokens:", encoded.tokens)
    print("ids:   ", encoded.ids)
assert local_tokenizer.encode("مرحبا بكم").tokens == ["[CLS]", "مرحبا", "بكم", "[SEP]"]
print("Local WordPiece demonstration=PASS")

أهلاً وسهلاً بكم في برنامج بيان!
tokens: ['[CLS]', 'أهلاً', 'وسهلاً', 'بكم', 'في', 'برنامج', 'بيان', '!', '[SEP]']
ids:    [2, 5, 6, 7, 8, 9, 10, 11, 3]
مرحباً بكم
tokens: ['[CLS]', '[UNK]', 'بكم', '[SEP]']
ids:    [2, 1, 7, 3]
Contact us at <EMAIL>
tokens: ['[CLS]', 'Contact', 'us', 'at', '<', 'EMAIL', '>', '[SEP]']
ids:    [2, 13, 14, 15, 16, 17, 18, 3]
للتجربة فقط: <PHONE>
tokens: ['[CLS]', '[UNK]', '[UNK]', '[UNK]', '<', '[UNK]', '>', '[SEP]']
ids:    [2, 1, 1, 1, 16, 1, 18, 3]
Natural language processing connects text and models.
tokens: ['[CLS]', 'Natural', 'language', 'processing', 'connects', 'text', 'and', 'models', '.', '[SEP]']
ids:    [2, 19, 20, 21, 22, 23, 24, 25, 26, 3]
Local WordPiece demonstration=PASS


In [ ]:
# Distinction: clitic test with "وبالخدمة"

clitic_text = "وبالخدمة الإلكترونية الجديدة"
clitic_encoded = local_tokenizer.encode(clitic_text)

print("Text:", clitic_text)
print("Tokens:", clitic_encoded.tokens)
print("IDs:", clitic_encoded.ids)

assert len(clitic_encoded.tokens) > 0
print("Clitic test=PASS")

Text: وبالخدمة الإلكترونية الجديدة
Tokens: ['[CLS]', '[UNK]', '[UNK]', '[UNK]', '[SEP]']
IDs: [2, 1, 1, 1, 3]
Clitic test=PASS


Distinction Challenge — Clitic Test

This Distinction challenge is documented in Lab 1 because it evaluates subword tokenisation behavior on the Arabic clitic “وبالخدمة”.

What did the test show?
The local tokenizer processed “وبالخدمة الإلكترونية الجديدة”, but produced [UNK] tokens because the local educational vocabulary does not contain all of these tokens. This does not mean the Arabic text is incorrect; it shows a limitation of the local tokenizer vocabulary.

تحدي Distinction — اختبار الـ Clitic

تم توثيق هذا التحدي في Lab 1 لأنه يختبر سلوك الـSubword Tokenisation مع الـClitic العربي “وبالخدمة”.

ماذا أظهر الاختبار؟
عالج الـTokenizer المحلي الجملة “وبالخدمة الإلكترونية الجديدة”، لكنه أعطى رموز [UNK] لأن الـVocabulary التعليمي المحلي لا يحتوي على جميع هذه الـTokens. هذا لا يعني أن النص العربي خاطئ، بل يوضح محدودية الـVocabulary في الـTokenizer المحلي

**المتوقع / Expected:** Tokens وIDs لكل نص. ظهور `[UNK]` يعني أن قاموس هذا المرمّز التعليمي لا يعرف الجزء؛ لا يعني أن النص خاطئ.

### مقياسان قبل اختيار المرمّز / Two pre-selection metrics

\[
\text{Token Fertility}=\frac{\text{عدد الرموز الفرعية}}{\text{عدد الكلمات التقريبية}}
\]

\[
\text{Truncation Rate}=\frac{\text{النصوص التي تتجاوز الحد}}{\text{كل النصوص}}
\]

خصوبة أعلى تعني عادة تسلسلاً أطول وكلفة أكبر، لكنها ليست وحدها حكمًا على جودة النموذج.

In [ ]:
def word_count(text: str) -> int:
    return max(1, len(text.split()))


def token_fertility(tokenizer: Tokenizer, text: str) -> float:
    tokens = tokenizer.encode(text).tokens
    content = [t for t in tokens if t not in {"[CLS]", "[SEP]", "[PAD]"}]
    return len(content) / word_count(text)


def truncation_rate(tokenizer: Tokenizer, texts: Iterable[str], max_length: int) -> float:
    texts = list(texts)
    if not texts:
        raise ValueError("texts must not be empty")
    return sum(len(tokenizer.encode(t).ids) > max_length for t in texts) / len(texts)

model_texts = [record.model_text for record in records]
fertilities = [token_fertility(local_tokenizer, text) for text in model_texts]
rate_at_10 = truncation_rate(local_tokenizer, model_texts, max_length=10)
print("fertility per sample:", [round(x, 2) for x in fertilities])
print("mean fertility:", round(float(np.mean(fertilities)), 2))
print("truncation rate @10:", f"{rate_at_10:.0%}")
assert all(x > 0 for x in fertilities)
assert 0.0 <= rate_at_10 <= 1.0
print("Tokenisation metrics=PASS")

fertility per sample: [1.17, 1.0, 1.5, 2.0, 1.14]
mean fertility: 1.36
truncation rate @10: 0%
Tokenisation metrics=PASS


In [ ]:
# Change from Core: separate Arabic and English texts before measuring fertility.
arabic_texts = [
    text for text in model_texts
    if re.search(r"[\u0600-\u06FF]", text)
]

english_texts = [
    text for text in model_texts
    if not re.search(r"[\u0600-\u06FF]", text)
]

arabic_fertilities = [
    token_fertility(local_tokenizer, text)
    for text in arabic_texts
]

english_fertilities = [
    token_fertility(local_tokenizer, text)
    for text in english_texts
]

print("Arabic fertility per sample:",
      [round(x, 2) for x in arabic_fertilities])
print("Arabic mean fertility:",
      round(float(np.mean(arabic_fertilities)), 2))

print("English fertility per sample:",
      [round(x, 2) for x in english_fertilities])
print("English mean fertility:",
      round(float(np.mean(english_fertilities)), 2))

Arabic fertility per sample: [1.17, 1.0, 2.0]
Arabic mean fertility: 1.39
English fertility per sample: [1.5, 1.14]
English mean fertility: 1.32


In [ ]:
# Measure truncation separately for Arabic and English at max_length 10 and 12.
arabic_rate_at_10 = truncation_rate(
    local_tokenizer, arabic_texts, max_length=10
)
arabic_rate_at_12 = truncation_rate(
    local_tokenizer, arabic_texts, max_length=12
)

english_rate_at_10 = truncation_rate(
    local_tokenizer, english_texts, max_length=10
)
english_rate_at_12 = truncation_rate(
    local_tokenizer, english_texts, max_length=12
)

print("Arabic truncation rate @10:", f"{arabic_rate_at_10:.0%}")
print("Arabic truncation rate @12:", f"{arabic_rate_at_12:.0%}")

print("English truncation rate @10:", f"{english_rate_at_10:.0%}")
print("English truncation rate @12:", f"{english_rate_at_12:.0%}")

Arabic truncation rate @10: 0%
Arabic truncation rate @12: 0%
English truncation rate @10: 0%
English truncation rate @12: 0%


## 3) Padding وTruncation ثم Embeddings

- **Padding:** إضافة `[PAD]` لتوحيد طول الدفعة؛ قناع الانتباه يضع 0 عند الحشو.
- **Truncation:** قص النص عند حد؛ قد يحذف المعلومة الحاسمة.
- **Token ID:** فهرس صحيح.
- **Embedding:** متجه أعداد حقيقية يُتعلم أثناء التدريب.

المتجهات التالية عشوائية للشرح فقط وليست تضمينات لغوية مدربة.

In [ ]:
MAX_LENGTH = 12
local_tokenizer.enable_truncation(max_length=MAX_LENGTH)
local_tokenizer.enable_padding(length=MAX_LENGTH, pad_id=vocab["[PAD]"], pad_token="[PAD]")
batch = [local_tokenizer.encode(text) for text in model_texts[:2]]
input_ids = np.array([item.ids for item in batch], dtype=np.int64)
attention_mask = np.array([item.attention_mask for item in batch], dtype=np.int64)
embedding_table = rng.normal(0.0, 0.02, size=(len(vocab), 8))
embeddings = embedding_table[input_ids]
print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("embeddings shape:", embeddings.shape)
print("first attention mask:", attention_mask[0].tolist())
assert input_ids.shape == (2, MAX_LENGTH)
assert attention_mask.shape == input_ids.shape
assert embeddings.shape == (2, MAX_LENGTH, 8)
print("IDs-to-embeddings pipeline=PASS")

input_ids shape: (2, 12)
attention_mask shape: (2, 12)
embeddings shape: (2, 12, 8)
first attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
IDs-to-embeddings pipeline=PASS


In [ ]:
# Distinction challenge: repeat toy embeddings with three seeds

seeds = [1, 2, 3]

for seed in seeds:
    run_rng = np.random.default_rng(seed)
    run_embedding_table = run_rng.normal(
        0.0, 0.02, size=(len(vocab), 8)
    )
    run_embeddings = run_embedding_table[input_ids]

    print(f"Seed {seed}")
    print("embeddings shape:", run_embeddings.shape)
    print("first embedding:", run_embeddings[0, 0].round(4))
    print()

print("Repeated toy embedding runs=PASS")

Seed 1
embeddings shape: (2, 12, 8)
first embedding: [ 0.0008 -0.0058 -0.0156 -0.0051  0.0002 -0.0055  0.0259  0.0201]

Seed 2
embeddings shape: (2, 12, 8)
first embedding: [ 0.0109 -0.0121  0.0025 -0.0178  0.0168  0.0038  0.0066  0.0082]

Seed 3
embeddings shape: (2, 12, 8)
first embedding: [ 0.0096 -0.0048  0.0192 -0.004   0.0005  0.0309  0.0109 -0.0101]

Repeated toy embedding runs=PASS


Distinction Challenge — Repeated Toy Embeddings

What changed and what stayed the same? The embedding values changed across Seeds 1, 2, and 3 because each seed generated different random values. However, the embedding shape stayed the same at (2, 12, 8) in all three runs. This shows that the random initialization changes the values, while the structure remains unchanged.

تحدي Distinction — تكرار Toy Embeddings

ما الذي تغيّر وما الذي بقي ثابتًا؟ تغيّرت قيم الـ Embeddings بين Seed 1 وSeed 2 وSeed 3 لأن كل Seed ولّد قيمًا عشوائية مختلفة. لكن شكل الـ Embeddings بقي ثابتًا (2, 12, 8) في التشغيلات الثلاثة. وهذا يوضح أن التهيئة العشوائية تغيّر القيم، بينما تبقى البنية ثابتة

**المتوقع / Expected:** الأشكال `(2, 12)` و`(2, 12, 8)` ثم رسالة نجاح. الحشو لا يحمل معنى؛ قناع الانتباه يمنع النموذج من معاملته كنص.

## Explore — مقارنة اختيارية مع mBERT

هذه الخلية تحاول تنزيل مرمّز `google-bert/bert-base-multilingual-cased` من Hugging Face. الاستخدام مجاني، لكن التنزيل يحتاج شبكة وقد يتعذر مؤقتًا. فشلها **لا يمنع اجتياز Core**.

In [ ]:
try:
    from transformers import AutoTokenizer
    hf_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased", use_fast=True)
    comparison_text = "معالجة اللغة الطبيعية مفيدة Natural language processing is useful"
    hf_tokens = hf_tokenizer.tokenize(comparison_text)
    print("mBERT tokens:", hf_tokens)
    print("mBERT token count:", len(hf_tokens))
    print("Fast tokenizer:", hf_tokenizer.is_fast)
except Exception as exc:
    print("Optional mBERT download unavailable; continue with Core.")
    print("Reason:", type(exc).__name__)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

mBERT tokens: ['مع', '##الجة', 'اللغة', 'الطبيعية', 'م', '##فيد', '##ة', 'Natural', 'language', 'processing', 'is', 'useful']
mBERT token count: 12
Fast tokenizer: True


In [ ]:
# Distinction: compare documented mBERT tokenizer with the same fertility metric

word_count = len(comparison_text.split())

mbert_token_count = len(hf_tokens)
mbert_fertility = mbert_token_count / word_count

local_tokens = local_tokenizer.encode(comparison_text).tokens
local_token_count = len(local_tokens) - 2  # exclude [CLS] and [SEP]
local_fertility = local_token_count / word_count

print("Approximate word count:", word_count)
print("Local tokenizer token count:", local_token_count)
print("Local tokenizer fertility:", round(local_fertility, 2))
print("mBERT token count:", mbert_token_count)
print("mBERT fertility:", round(mbert_fertility, 2))

print("Documented tokenizer comparison=PASS")

Approximate word count: 9
Local tokenizer token count: 10
Local tokenizer fertility: 1.11
mBERT token count: 12
mBERT fertility: 1.33
Documented tokenizer comparison=PASS


Distinction Challenge — Documented Tokenizer Comparison

What did the comparison show? The local tokenizer and mBERT were compared using the same Token Fertility metric on the same text. The local tokenizer had a fertility of 1.11, while mBERT had a fertility of 1.33. This comparison describes tokenization behavior only and does not mean that one tokenizer is better for the final task.

تحدي Distinction — مقارنة Tokenizer موثّق

ماذا أظهرت المقارنة؟ تمت مقارنة الـLocal Tokenizer وmBERT باستخدام نفس مقياس Token Fertility وعلى نفس النص. كانت قيمة الـFertility للـLocal Tokenizer تساوي 1.11، بينما كانت لـmBERT تساوي 1.33. هذه المقارنة تصف سلوك الترميز فقط، ولا تعني أن أحد الـTokenizers أفضل للمهمة النهائية

## 4) قرار موثق / Documented decision

| البند / Criterion | ملاحظتي / My evidence |
|---|---|
| Profile المطبق |NFC + whitespace cleanup + tatweel removal|
| مثال قبل/بعد |مرحبـاً بكم / مرحباً بكم|
| متوسط Token Fertility |Arabic: 1.39 / English:1.32|
| Truncation Rate والحد |Arabic @10: 0% / @12: 0% English @10: 0% / @12: 0%|
| خطر محتمل على العربية |Arabic Subword Fragmentation may complicate NER alignment|
| قراري والسبب |Keep the current tokenizer as the baseline because the Arabic and English metrics pass; mBERT was explored(12 tokens), but further comparison is needed before any change.|

### مستويات التحدي

- **Core:** شغّل Core واشرح مثالًا قبل/بعد.
- **Explore:** قارن المحلي مع mBERT على 5 جمل اصطناعية.
- **Distinction:** قارن مرمزين مناسبين وناقش الخصوبة والقطع والزمن دون اعتبار مقياس واحد إثباتًا للأفضلية.

In [ ]:
core_checks = {
    "unicode": unicode_rows[0]["code_point"] == "U+0623",
    "spacy_sentence_pipeline": all(len(value) == 2 for value in sentence_splits.values()),
    "raw_copy_preserved": records[2].raw_text == samples[2],
    "pii_masked": "<EMAIL>" in records[2].model_text and "<PHONE>" in records[3].model_text,
    "token_metrics": all(x > 0 for x in fertilities) and 0 <= rate_at_10 <= 1,
    "embedding_shape": embeddings.shape == (2, MAX_LENGTH, 8),
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY1_NOTEBOOK1_CORE=PASS")

unicode: PASS
spacy_sentence_pipeline: PASS
raw_copy_preserved: PASS
pii_masked: PASS
token_metrics: PASS
embedding_shape: PASS
DAY1_NOTEBOOK1_CORE=PASS


In [ ]:
!git clone https://github.com/almiyead-rgb/bayan-applied-nlp-course.git

Cloning into 'bayan-applied-nlp-course'...
remote: Enumerating objects: 384, done.
remote: Counting objects: 100% (384/384), done.
remote: Compressing objects: 100% (348/348), done.
remote: Total 384 (delta 152), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (384/384), 582.80 KiB | 5.77 MiB/s, done.
Resolving deltas: 100% (152/152), done.


In [ ]:
!cd bayan-applied-nlp-course && PYTHONPATH=src pytest -q tests/test_day1_preprocessing.py tests/test_day1_tokenization.py

......                                                                   [100%]
6 passed in 0.15s


In [ ]:
!cat bayan-applied-nlp-course/student-starter/DECISIONS.md

# DECISIONS — Bayan

> انسخ القالب إلى `DECISIONS.md`. أضف قرارًا جديدًا لكل تغيير يؤثر في البيانات أو الجودة أو الخدمة.  
> Copy to `DECISIONS.md`. Add one record for each material data, quality, or serving decision.

## Decision D-001 — FILL_ME

- **Date:** FILL_ME
- **Gate:** A / B / C / D / E — FILL_ME
- **Status:** proposed / accepted / rejected / superseded — FILL_ME
- **Owner:** FILL_ME

### Context | السياق

FILL_ME: ما المشكلة أو القيد؟ ما الذي بقي ثابتًا؟

### Options considered | البدائل

| Option | Benefit | Cost/risk | Evidence |
|---|---|---|---|
| A | FILL_ME | FILL_ME | FILL_ME |
| B | FILL_ME | FILL_ME | FILL_ME |

### Decision | القرار

FILL_ME

### Evidence | الدليل

- Report/test/commit: FILL_ME
- Metric and result label: FILL_ME
- Slice or failure considered: FILL_ME

### Consequences and rollback | الأثر والرجوع

- Positive consequence: FILL_ME
- Limitation/new risk: FILL_ME
- Rollback trigger: FILL_ME
- Rollback path: FILL_ME

---

## قرارات إلزامية قبل Gate E

-

## Decision D-001 — Tokenizer and max length

- **Date:** 2026-09-02
- **Gate:** A
- **Status:** accepted
- **Owner:** Reema Alzurayqi

### Context | السياق
The tokenizer and max length need to be documented for the Day 1 baseline. The current preprocessing profile and synthetic corpus slice remain unchanged.

### Options considered | البدائل

| Option | Benefit | Cost/risk | Evidence |
|---|---|---|---|
| A | Keep the current tokenizer as the baseline | Simple and already tested on the current corpus | Arabic subword fragmentation may affect later NER alignment | Arabic fertility: 1.39 / English fertility: 1.32; truncation: 0% at max_length 10 and 12 |
| B | Change to mBERT tokenizer | Provides a multilingual tokenizer for Arabic and English | More comparison is needed before changing the baseline | mBERT was explored and produced 12 tokens on the tested mixed sentence |

### Decision | القرار
Keep the current tokenizer as the baseline with max_length=12. mBERT was explored, but further comparison is needed before any change.

### Evidence | الدليل
- Report/test/commit: DAY1_NOTEBOOK1_CORE=PASS; 6 pytest tests passed
- Metric and result label: Arabic fertility=1.39; English fertility=1.32; Arabic/English truncation rate=0% at max_length 10 and 12
- Slice or failure considered: Arabic and English synthetic samples; Arabic subword fragmentation

### Consequences and rollback | الأثر والرجوع
- Positive consequence: Keeps a tested baseline for the next stages.
- Limitation/new risk: Arabic subword fragmentation may complicate NER alignment.
- Rollback trigger: Later comparison shows a clear improvement with another tokenizer.
- Rollback path: Replace the tokenizer and its paired checkpoint after measured comparison.

## Decision D-002 — Arabic preprocessing profile

- **Date:** 2026-09-02
- **Gate:** A
- **Status:** accepted
- **Owner:** Reema Alzurayqi

### Context | السياق

A conservative Arabic preprocessing profile is needed for the Day 1 baseline. The original display copy should remain unchanged, while the model copy can apply limited normalization.

### Options considered | البدائل

| Option | Benefit | Cost/risk | Evidence |
|---|---|---|---|
| A | Use NFC + whitespace cleanup + tatweel removal | Simple and conservative; keeps the original Arabic form mostly intact | Some Arabic variation remains unchanged | The before/after example changed `مرحبــاً بكم` to `مرحباً بكم` while preserving the diacritics |
| B | Add stronger normalization such as Alef normalization or diacritic removal | May reduce surface variation | May remove distinctions or change the original form more than needed | These options are available in the notebook but were not enabled in the baseline |

### Decision | القرار

Use NFC + whitespace cleanup + tatweel removal as the Arabic preprocessing baseline. Keep stronger normalization options disabled for now.

### Evidence | الدليل

- Report/test/commit: Two-copy preprocessing contract=PASS; SPACY_SENTENCE_PIPELINE=PASS; DAY1_NOTEBOOK1_CORE=PASS; 6 pytest tests passed
- Metric and result label: Arabic fertility=1.39; English fertility=1.32
- Slice or failure considered: Arabic synthetic samples and the protected model-text pipeline

### Consequences and rollback | الأثر والرجوع

- Positive consequence: Keeps preprocessing simple and preserves most of the Arabic text form.
- Limitation/new risk: Some Arabic spelling variation remains and may affect later tasks.
- Rollback trigger: Later evaluation shows that stronger normalization gives a clear measured improvement.
- Rollback path: Enable one additional normalization option, measure the effect, and compare it with the baseline.

## قرارات إلزامية قبل Gate E

- [x] tokenizer + max length.
- [x] Arabic preprocessing profile.
- [ ] task model/baseline and split.
- [ ] semantic encoder/index/k/threshold.
- [ ] metric/slices/error priorities.
- [ ] performance budget.
- [ ] ONNX/INT8 adopt or reject.
- [ ] served artefact + preprocessing/label versions.

## نقطة GitHub / GitHub checkpoint

1. من Colab: **File → Save a copy in GitHub**.
2. احفظ النسخة في مستودعك العام داخل `notebooks/`.
3. Commit: `feat(day1): complete preprocessing and tokenisation lab`.
4. أضف في `README.md`: Profile، الخصوبة، معدل القطع، وقرارك.
5. لا تحفظ مفاتيح أو بيانات حقيقية أو مخرجات تكشف معلومات شخصية.

**دليل الاكتمال:** `DAY1_NOTEBOOK1_CORE=PASS` + رابط Commit عام + جدول القرار.